<center><h1>Chen_Jessica_HW3</h1></center>
<br>
<br>

Name: Jessica Chen
<br>
Github Username: jc6006 
<br>
USC ID: 9760667304

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [62]:
import io
import numpy as np
import pandas as pd

Get the AReM Data Set

In [63]:
cols=["time","avg_rss12","var_rss12","avg_rss13","var_rss13","avg_rss23","var_rss23"]
counts={"bending1":7,"bending2":6,"cycling":15,"lying":15,"sitting":15,"standing":15,"walking":15}

def open_file(path):
    with open(path) as f:
        lines=[l.strip().rstrip(",") for l in f if not l.startswith("#") and l.strip()]
    return pd.read_csv(io.StringIO("\n".join(lines)),header=None,names=cols,sep=r"[,\s]+",engine="python")

### (b) Test and Train Data

In [64]:
train,test=[],[]
for activity,n in counts.items():
    test_ids={1,2} if activity in ("bending1","bending2") else {1,2,3}
    for i in range(1,n+1):
        path=f"../data/AReM/{activity}/dataset{i}.csv"
        df=open_file(path)
        item={"activity":activity,"file":f"dataset{i}.csv","data":df}
        if i in test_ids:
            test.append(item)
        else:
            train.append(item)

### (c) Feature Extraction

#### i. Research

Mean, median, min, max, range, std, variance, Q1, Q3, IQR, RMS, skewness, kurtosis, and auto-correlation are some common ones.

https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html

#### ii. Extraction

In [65]:
time_cols=cols[1:]
feat_names=["min","max","mean","median","std","1st quart","3rd quart"]
def extract(d):
    row={}
    for k,c in enumerate(time_cols,start=1):
        s=d[c]
        row[f"min{k}"]=s.min()
        row[f"max{k}"]=s.max()
        row[f"mean{k}"]=s.mean()
        row[f"median{k}"]=s.median()
        row[f"std{k}"]=s.std()
        row[f"1st quart{k}"]=s.quantile(0.25)
        row[f"3rd quart{k}"]=s.quantile(0.75)
    return row
    
def build_table(items):
    rows=[]
    for it in items:
        r=extract(it["data"])
        r["activity"]=it["activity"]
        r["file"]=it["file"]
        rows.append(r)
    return pd.DataFrame(rows)

train_df=build_table(train)
test_df=build_table(test)

#### iii. Standard Deviation

In [66]:
complete_df=pd.concat([train_df, test_df],ignore_index=True)
feature_cols=[c for c in complete_df.columns if c not in ("activity","file")]
feature_array=complete_df[feature_cols].to_numpy()
instances=feature_array.shape[0]
B=1000
rng=np.random.default_rng(0)
idx=rng.integers(0,instances,size=(B,instances))
boot_std=feature_array[idx].std(axis=1,ddof=1)
low,high=np.percentile(boot_std,[5,95],axis=0)
ci_df=pd.DataFrame({"std":feature_array.std(axis=0,ddof=1),"CI lower (5%)":low,"CI upper (95%)":high,}, index=feature_cols)
ci_df

,std,CI lower (5%),CI upper (95%)
min1,9.569975,8.288933,10.712363
max1,4.394362,3.359798,5.286204
mean1,5.335718,4.732774,5.858968
median1,5.440054,4.816189,5.979534
std1,1.772153,1.580998,1.950934
1st quart1,6.153590,5.614170,6.648465
3rd quart1,5.138925,4.379412,5.796135
min2,0.000000,0.000000,0.000000
max2,5.062729,4.657666,5.411793
mean2,1.574164,1.403705,1.705372


#### iv. Select Features

In [67]:
print(ci_df.loc[["min2","min4","min6"]])
print()
print(complete_df[["mean1","median1","1st quart1","3rd quart1","max1","std1"]].corr())
print()

           std  CI lower (5%)  CI upper (95%)
min2  0.000000            0.0        0.000000
min4  0.000000            0.0        0.000000
min6  0.045838            0.0        0.078476

               mean1   median1  1st quart1  3rd quart1      max1      std1
mean1       1.000000  0.978048    0.970171    0.955637  0.520980 -0.419841
median1     0.978048  1.000000    0.968993    0.913133  0.482380 -0.423381
1st quart1  0.970171  0.968993    1.000000    0.877026  0.385889 -0.547132
3rd quart1  0.955637  0.913133    0.877026    1.000000  0.636707 -0.170159
max1        0.520980  0.482380    0.385889    0.636707  1.000000  0.326911
std1       -0.419841 -0.423381   -0.547132   -0.170159  0.326911  1.000000



Median, max, and standard deviation. Min doesn't really vary across the time series. Mean, median, 1st quart, and 3rd quart in the first series seem to be closely correlated. 

## 2. ISLR 3.7.4

### (a) Linear Train

Cubic should be lower, as it'll fit the random error in the set of n=100 observations better and therefore lower training RSS. 

### (b) Linear Test

Linear should be lower, as cubic will likely overfit the errors in the set of observations and end up predicting worse. This will make cubic have higher test RSS than linear.

### (c) Not Linear Train

Cubic should be lower, as it'll be able to fit better to a non-linear relationship than linear would. Therefore it would have lower training RSS.

### (d) Not Linear Testing

We wouldn't be able to tell, it would depend on how non-linear the actual relationship is. If it is very non-linear cubic would have lower RSS, but if it is only slightly non-linear then linear would have lower test RSS.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 